# 05 — Feature Selection

**Objective:** Plan leakage-safe feature-selection experiments.  
**Owner:** Ilias El Hamri  
**Sprint:** 01  

> Leakage warning: select features independently inside each training fold.

## Project Setup

This cell locates the project root and loads the shared configuration.

- The project root is found by walking up from the current working directory until a folder containing `configs/config.yaml` is found. This ensures the notebook works regardless of where Jupyter is launched from.
- The project root is inserted into `sys.path` so that `src.*` imports resolve correctly.
- `load_config()` reads `configs/config.yaml` and returns a dictionary that contains shared settings such as `random_state`, output paths, and metric names.

In [ ]:
from pathlib import Path

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "configs" / "config.yaml").is_file())
import sys
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from src.config import load_config

config = load_config()
print(config["project"]["name"])
print(config["project"]["random_state"])

## Planned work

Later: define selection methods, nest them in validated pipelines, compare feature counts, and document stability and interpretation.

## Data Loading

This cell loads the Santander Customer Transaction dataset from OpenML using the shared utility.

- `optimize_memory=True` converts numeric features from `float64` to `float32`, reducing memory usage by roughly half while preserving all structural invariants.
- The function returns three objects:
  - `X` — a DataFrame of 200 anonymised feature columns.
  - `y` — a binary target Series (`"0"` / `"1"`).
  - `metadata` — a dictionary with dataset description and provenance information.

In [ ]:
from src.data import load_dataset

X, y, metadata = load_dataset(optimize_memory=True)

## Pipeline Construction

This cell builds the full feature-selection pipeline using scikit-learn's `Pipeline` to prevent data leakage.

### Feature selector

- `SelectFromModel` wraps an L1-penalised Logistic Regression as the embedded selector.
- L1 regularisation shrinks uninformative feature coefficients to **exactly zero**, so `SelectFromModel` automatically discards those features.
- `C=0.1` applies strong regularisation to aggressively zero out weak features.
- `solver='saga'` is required for L1 penalty support in scikit-learn.

### Final classifier

- The final classifier uses L2 regularisation (`lbfgs` solver), which is numerically stable and well-suited for high-dimensional data after selection has been applied.

### Why a Pipeline?

- Bundling all three steps — scaling → selection → classification — inside a `Pipeline` is mandatory.
- `evaluate_model_cv` refits the entire pipeline independently inside each cross-validation training fold, so no information from the validation or test partitions ever leaks into the scaler or the selector.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression

feature_selector = SelectFromModel(
    LogisticRegression(penalty="l1", solver="saga", C=0.1, random_state=config["project"]["random_state"])
)

classifier = LogisticRegression(
    penalty="l2", solver="lbfgs", C=1.0, max_iter=1000, random_state=config["project"]["random_state"])

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("selector", feature_selector),
    ("classifier", classifier)
])

## Cross-Validation Evaluation

This cell evaluates the pipeline using the shared project infrastructure.

### What `evaluate_model_cv` does

- Applies 5-fold stratified cross-validation on **training data only**.
- The final test partition is never passed and remains entirely closed.
- Returns two objects:
  - `fold_results` — a DataFrame with per-fold metrics (ROC-AUC, Average Precision, F1, etc.).
  - `summary` — a dictionary with aggregate scores, estimator parameters, and experiment metadata.

### Parameters

- `experiment_id="M03-FS-001"` — unique identifier for this experiment (Member 03, Feature Selection, run 001).
- `member="Member 03"` — identifies Ilias El Hamri as the author.
- `branch="feature/feature-selection"` — Git branch this notebook belongs to.

### Output

- The primary metric (ROC-AUC) mean and standard deviation across the five folds are printed. The standard deviation indicates how stable the model is across different data subsets.

In [ ]:
from src.evaluation import evaluate_model_cv

fold_results, summary = evaluate_model_cv(
    estimator=pipeline,
    X=X,
    y=y,
    model_name="L1-FeatureSelection + LogisticRegression",
    experiment_id="M03-FS-001",
    member="Member 03",
    branch="feature/feature-selection"
)
print(f"Primary Metric ({summary['primary_metric']}): {summary['primary_score_mean']:.4f} +/- {summary['primary_score_std']:.4f}")